# Assignment 5 - Part 1: Duck vs Chicken Classification

Fine-tune a pre-trained CNN (ResNet18) to classify duck vs chicken images.

**Data layout expected:**
```
data/
  train/
    chicken/*.jpg
    duck/*.jpg
  val/
    chicken/*.jpg
    duck/*.jpg
```

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
from sklearn.metrics import classification_report

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cpu


## Data loading and transforms

Resize to 224x224 (what ResNet expects) and normalize with ImageNet stats.

In [2]:
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder('data/train', transform=train_tf)
val_ds = datasets.ImageFolder('data/val', transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)

class_names = train_ds.classes
print('Classes:', class_names)
print('Train size:', len(train_ds), '| Val size:', len(val_ds))

Classes: ['chicken', 'duck']
Train size: 897 | Val size: 161


## Load pre-trained ResNet18 and swap the final layer

Freeze all layers, then replace the final fully-connected layer with a new one for 2 classes. Only the new layer will train.

In [3]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /Users/codula/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


  0%|          | 0.00/44.7M [00:00<?, ?B/s]

  0%|          | 128k/44.7M [00:00<00:41, 1.11MB/s]

  1%|          | 512k/44.7M [00:00<00:19, 2.34MB/s]

  2%|▏         | 1.00M/44.7M [00:00<00:14, 3.09MB/s]

...

 96%|█████████▋| 43.0M/44.7M [00:11<00:00, 4.00MB/s]

 97%|█████████▋| 43.5M/44.7M [00:11<00:00, 4.00MB/s]

 99%|█████████▊| 44.0M/44.7M [00:11<00:00, 3.99MB/s]

100%|█████████▉| 44.5M/44.7M [00:11<00:00, 4.01MB/s]

100%|██████████| 44.7M/44.7M [00:11<00:00, 3.93MB/s]

## Train

In [4]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)

epochs = 5
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f'Epoch {epoch+1}/{epochs} - loss: {running_loss/len(train_loader):.4f}')

Epoch 1/5 - loss: 0.3615


Epoch 2/5 - loss: 0.2042


Epoch 3/5 - loss: 0.1758


Epoch 4/5 - loss: 0.1761


Epoch 5/5 - loss: 0.1820


## Evaluate and print classification report

In [5]:
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()
        y_true.extend(labels.numpy())
        y_pred.extend(preds)

print(classification_report(y_true, y_pred, target_names=class_names))

              precision    recall  f1-score   support

     chicken       0.98      0.83      0.90        52
        duck       0.92      0.99      0.96       109

    accuracy                           0.94       161
   macro avg       0.95      0.91      0.93       161
weighted avg       0.94      0.94      0.94       161

